In [2]:
import math 
import random
from collections import Counter, defaultdict
import nltk

## Load and split the data

In [3]:
print('Download data')
nltk.download("brown", quiet=True)
nltk.download("universal_tagset", quiet=True)
from nltk.corpus import brown

Download data


In [4]:
# grab sentences and drop empty ones
all_sentences = [s for s in brown.tagged_sents(tagset = 'universal') if len(s)>0]

In [5]:
# sequential split data into 80/20 as required
split_point = int(len(all_sentences)*0.8)
train_data = all_sentences[:split_point]
test_data = all_sentences[split_point:]

print(f'Train sentences: {len(train_data)}, Test sentences: {len(test_data)}')

Train sentences: 45872, Test sentences: 11468


## MLE counting and Math matric build

In [6]:
alpha = 1       # laplace add-one-smoothing
eps = 1e-12     #safty epsilon to prevent log(0) domain error

In [7]:
#raw frequency count

count_start = Counter() #C_start(t_i)
count_trans = defaultdict(Counter) # C(t_i,t_j)
count_emit = defaultdict(Counter) #C(t,w)
count_tag = Counter() #C(t) count of all tag
vocab = set() #the word that already known

In [8]:
#start counting 

for sentence in train_data:
    first_word, first_tag = sentence [0]
    count_start[first_tag] +=1

    for i in range(len(sentence)):
        word, tag = sentence[i]
        vocab.add(word)
        count_tag[tag] +=1
        count_emit[tag][word] +=1

        if i < len(sentence) -1:
            next_word, next_tag = sentence[i+1]
            count_trans[tag][next_tag] +=1

In [9]:
#final sets and vocabulary sizes
tags = list(count_tag.keys())       #total number of each tags
N_sentences = len(train_data)   #total number of sentence
T_size = len(tags)      #total number of tag
V_size = len(vocab)     #total number of word

In [10]:
#build martrices
pi_matrix= {}
A_matrix = defaultdict(dict)
B_matrix = defaultdict(dict)
#当维特比算法遇到一个在训练集中从未见过的单词（OOV, Out-of-Vocabulary）时，提供一个合法的概率参考值。
oov_matrix = {}         #break up score if word isn't in vocabulary

In [11]:
for t in tags:
    #initial
    prob_pi = (count_start[t] + alpha)/(N_sentences + alpha * T_size) #add-one-smoothing
    pi_matrix[t] = math.log(prob_pi + eps) #log space + eps

    #OOV Fallback: log( alpha / (C(t) + alpha * |V|) )
    prob_oov = alpha / (count_tag[t] + alpha * V_size)
    oov_matrix[t] = math.log(prob_oov + eps)

    #transition
    total_out_transitions = sum(count_trans[t].values())
    for next_t in tags:
        prob_a = (count_trans[t][next_t]+ alpha)/(total_out_transitions+ alpha * T_size)
        A_matrix[t][next_t] = math.log(prob_a +eps)
    
    #emission
    for w in vocab:
        prob_b = (count_emit[t][w] + alpha)/(count_tag[t]+ alpha * V_size)
        B_matrix[t][w] = math.log(prob_b + eps)

print(f'HMM tables generated successfully ')

HMM tables generated successfully 


## Log-Space Viterbi Decoder

In [12]:
def run_viterbi(words_list):
    if not words_list:
        return[] #to make sure that even it doesn't have any sentence it will still return []
    
    #trellis dictiomaries to hold values at each step
    viterbi_trellis = [] #viterbi_trellis[t] 存储的是句子中第 t 个单词在所有可能词性标签（POS tags）下的最大对数概率。
    backpointers =[] #backpointers[t][tag] 存储的是：如果当前第 t 个词被标注为 tag，那么上一个词（第 t-1 个词）最可能的标签是什么？

    #base case: t=0
    w0 = words_list[0]
    t0_probs = {}
    t0_bp = {}

    # use b matrix score, fallback to pre-calculated oov score if word is unknown
    for t in tags:
        b_score = B_matrix[t][w0] if w0 in vocab else oov_matrix[t]
        t0_probs[t] = pi_matrix[t] + b_score
        t0_bp[t] =None

    viterbi_trellis.append(t0_probs)
    backpointers.append(t0_bp)

    # inductive step: t > 0
    for step in range(1, len(words_list)):
        w_curr = words_list[step]
        step_probs = {}
        step_bp = {}

        for curr_tag in tags:
            b_score = B_matrix[curr_tag][w_curr] if w_curr in vocab else oov_matrix[curr_tag]

            # manual loop to find max previous path - very student-like
            best_val = -float("inf")
            best_prev = None

            for prev_tag in tags:
                # log additions replace multiplications
                score = viterbi_trellis[step - 1][prev_tag] + A_matrix[prev_tag][curr_tag] + b_score
                if score > best_val:
                    best_val = score
                    best_prev = prev_tag

            step_probs[curr_tag] = best_val
            step_bp[curr_tag] = best_prev

        viterbi_trellis.append(step_probs)
        backpointers.append(step_bp)

    # trace back the path from the end
    best_last_val = -float("inf")
    best_last_tag = None
    for t in tags:
        if viterbi_trellis[-1][t] > best_last_val:
            best_last_val = viterbi_trellis[-1][t]
            best_last_tag = t

    final_path = [best_last_tag]
    for step in range(len(words_list) - 1, 0, -1):
        best_last_tag = backpointers[step][best_last_tag]
        final_path.insert(0, best_last_tag)

    return final_path

## Evaluation

In [13]:
def check_accuracy(dataset):
    correct = 0
    total = 0
    for sent in dataset:
        words = [pair[0] for pair in sent]
        true_tags = [pair[1] for pair in sent]

        pred_tags = run_viterbi(words)
        for r, p in zip(true_tags, pred_tags):
            if r==p:
                correct +=1
            total +=1
    return correct/total if total > 0 else 0

In [14]:
test_acc = check_accuracy(test_data)
print(f'Accuracy on test set: {test_acc * 100:.2f}%')

Accuracy on test set: 92.47%


## Error case study

In [15]:
print(f'Looking for 3 error case study sentences: ')

error_count = 0  # 初始化计数器
for sent in test_data:
    words = [pair[0] for pair in sent]
    true_tags = [pair[1] for pair in sent]
    pred_tags = run_viterbi(words)

    # 筛选：预测有误且长度在 5 到 10 之间的句子
    if pred_tags != true_tags and 5 <= len(words) <= 10:
        error_count += 1
        
        print(f"\n--- Error Case #{error_count} ---")
        print(f"Sentence: {' '.join(words)}")
        print("Word | True Tag | Predicted Tag")
        
        for w, true_t, pred_t in zip(words, true_tags, pred_tags):
            flag = " <-- ERROR" if true_t != pred_t else ""
            print(f"{w} | {true_t} | {pred_t}{flag}")
    
        print("\n")
        
        if error_count >= 3:
            break


Looking for 3 error case study sentences: 

--- Error Case #1 ---
Sentence: What's below the water-line interests me also .
Word | True Tag | Predicted Tag
What's | PRT | PRT
below | ADP | ADP
the | DET | DET
water-line | NOUN | ADJ <-- ERROR
interests | VERB | NOUN <-- ERROR
me | PRON | PRON
also | ADV | ADV
. | . | .



--- Error Case #2 ---
Sentence: Red was small and fine-boned , like ivory-inlay .
Word | True Tag | Predicted Tag
Red | NOUN | PRON <-- ERROR
was | VERB | VERB
small | ADJ | ADJ
and | CONJ | CONJ
fine-boned | ADJ | NOUN <-- ERROR
, | . | .
like | ADP | ADP
ivory-inlay | NOUN | NOUN
. | . | .



--- Error Case #3 ---
Sentence: The Beech Pasture had suddenly become valuable .
Word | True Tag | Predicted Tag
The | DET | DET
Beech | NOUN | ADJ <-- ERROR
Pasture | NOUN | NOUN
had | VERB | VERB
suddenly | ADV | ADV
become | VERB | VERB
valuable | ADJ | ADJ
. | . | .


